# Segmentación y cuantificación de núcleos con `scikit-image` + `matplotlib`

El objetivo de este ejercicio es realizar un flujo de trabajo simple con pasos de preprocesamiento, segmentación, operaciones morfológicas y cuantificación utilizando modulos usuales de `scikit-image` y amigos en un jupyter notebook.
 
Incluye:
- preprocesamiento (mediana + corrección de fondo),
- umbralización (incluyendo `try_all_threshold`),
- etiquetado y separación opcional de objetos tocantes,
- cuantificación morfológica básica,
- controles interactivos con `ipywidgets`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from bffile import BioFile
from skimage import filters, morphology, measure, segmentation, feature, color, util, data
from skimage.restoration import rolling_ball
from scipy import ndimage as ndi

import ipywidgets as widgets
from ipywidgets import interact

In [ ]:
image_path = Path('./data/nuclei.nd2')

with BioFile(image_path) as bf:
    imagen = np.asarray(bf.as_array())

print('Forma:', imagen.shape, '| dtype:', imagen.dtype)

Noten que la imagen tiene 5 dimensiones, aunque las primeras solo tienen un solo stack, esto suele pasar al cargar imágenes. Numpy tiene una función para descartar todas las dimensiones que tienen un único elemento `numpy.sueeze`.

In [ ]:
imagen = np.squeeze(imagen)
print('Forma:', imagen.shape, '| dtype:', imagen.dtype)

plt.figure(figsize=(6, 6))
plt.imshow(imagen)
plt.title('Imagen cruda')
plt.axis('off')
plt.show()

Pueden buscar otras LuT y cambiarlas. Tambien pueden crear sus propias LuT. Consideren que por defecto, matplotlib realiza una interpolación y suavizado de las imágenes que puede alterar como la vemos, especialmente cuando hay altas componentes espaciales.

In [ ]:
HiLo_cmap = plt.get_cmap('grey').copy()
HiLo_cmap.set_under('blue')
HiLo_cmap.set_over('red')

In [ ]:
@interact(vmin=(0, np.max(imagen)), vmax=(0, np.max(imagen)))
def plot_raw(vmin=0, vmax=np.max(imagen)):
    plt.figure(figsize=(6, 6))
    plt.imshow(imagen, interpolation='none', cmap=HiLo_cmap, vmin=vmin, vmax=vmax)
    plt.title('Imagen cruda')
    plt.axis('off')
    plt.show()

## Preprocesamiento

Podemos seleccionar distintos algoritmos de preprocesamiento y evaluar el efecto de correrlos sobre la imagen cruda, o incluso concatenando operaciones. Sugiero visitar [la página de scikit-image](https://scikit-image.org/docs/stable/api/skimage.filters.html) para ver qué opciones hay y revisar la documentación.

Como en la celda anterior, uno puede usar el `interact` de `ipywidgets` para variar rápidamente los parámetros y explorar.

In [ ]:
@interact(radius=(0, 10))
def medain_filter(radius=3):
    suave = filters.median(imagen, morphology.disk(radius))
    
    fig, ax = plt.subplots(1, 2, figsize=(8, 6))
    ax[0].imshow(imagen); ax[0].set_title('Cruda'); ax[0].axis('off')
    ax[1].imshow(suave); ax[1].set_title(f'Mediana (r={radius})'); ax[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
radius = 3
suave = filters.median(imagen, morphology.disk(radius))

In [ ]:
@interact(
    radio_fondo=widgets.IntSlider(value=20, min=5, max=120, step=1),
)
def explorar_preprocesamiento(radio_fondo=20):
    fondo = rolling_ball(suave, radius=radio_fondo)
    corregida = suave - fondo
    corregida = np.clip(corregida, a_min=0, a_max=None)

    fig, ax = plt.subplots(1, 3, figsize=(8, 4))
    ax[0].imshow(imagen); ax[0].set_title('Cruda'); ax[0].axis('off')
    ax[1].imshow(fondo); ax[1].set_title(f'Fondo (rolling-ball r={radio_fondo})'); ax[1].axis('off')
    ax[2].imshow(corregida); ax[2].set_title('Corregida'); ax[2].axis('off')
    plt.tight_layout()
    plt.show()

Si hace falta, pueden hacer zoom en alguna región usando `plt.ylim` y `plt.xlim` (cuidado con el orden de los números que puede rotar la imagen) y también pueden reutilizar el LuT que armamos arriba.

In [ ]:
fondo = rolling_ball(suave, radius=40)
corregida = suave - fondo
corregida = np.clip(corregida, a_min=0, a_max=None)

## Umbralización

Una vez que este terminado el preprocesamiento, podemos indagar en qué métodos de umbralización funcionan mejor. No se preocupen si no están conformes con ninguno. Es muy difícil conseguir una segmentación perfecta para todos los casos y siempre se puede mejorar la segmentación con operaciones morfológicas o volver a corregir el preprocesamiento.

In [ ]:
fig, ax = filters.try_all_threshold(corregida, figsize=(14, 10), verbose=True)
plt.show()

Aunque no estemos del todo conformes podemos elegir el mejor de estos métodos y mejorar con postprocesamiento. ¿Se les ocurre alguna forma reproducible de tomar alguno de estos algoritmos y modificar el umbral? ¿Qué pasaría si tomamos el umbral hallado y lo multiplicamos por un valor entre 0 y 1, o uno mayor a 1?

In [ ]:
threshold = filters.threshold_otsu(corregida)
binaria = corregida > threshold

## Postprocesamiento

Dentro de las operaciones de postprocesamiento podemos hacer cosas como llenar agujeros con `fill_holes`, deshacernos de objetos pequeños (`remove_small_objects`) o grandes, aplicar operaciones morfológicas y separar objetos tocándose (`Watershed`).

In [ ]:
@interact(tamano_minimo=(0, 4000))
def remover_objetos_pequenos(tamano_minimo=20):
    binaria_filtrada = morphology.remove_small_objects(binaria, min_size=tamano_minimo)

    fig, ax = plt.subplots(1, 3, figsize=(8, 4))
    ax[0].imshow(corregida); ax[0].set_title('Corregida'); ax[0].axis('off')
    ax[1].imshow(binaria); ax[1].set_title('Máscara binaria'); ax[1].axis('off')
    ax[2].imshow(binaria_filtrada + binaria); ax[2].set_title('Máscara filtrada'); ax[2].axis('off')
    plt.tight_layout()
    plt.show()

Agreguen celdas y prueben distintas operaciones y combinaciones aca!

### Watershed

Seria interesante implementar watershed para ver si logramos separar los núcleos que están juntos.

Primero necesitamos una función que nos devuelva la distancia de cada pixel al borde de la máscara.

In [ ]:
distancia = ndi.distance_transform_edt(binaria)

plt.imshow(distancia)
plt.axis('off')
plt.show()

A continuacion necesitamos generar las semillas del watershed.

In [ ]:
coords = feature.peak_local_max(
    distancia,
    labels=binaria,
    min_distance=20,
    exclude_border=False
)

plt.imshow(distancia)
plt.scatter(coords[:, 1], coords[:,0], marker='x', color='red', s=2)
plt.axis('off')
plt.show()

Teniendo todas las semillas, podemos correr el algoritmo con todo lo que encontramos.

In [ ]:
marcadores_mask = np.zeros_like(binaria, dtype=bool)
marcadores_mask[tuple(coords.T)] = True
marcadores_mask = measure.label(marcadores_mask)
etiquetas = segmentation.watershed(-distancia, marcadores, mask=binaria)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 4))
ax[0].imshow(corregida); ax[0].set_title('Preprocesada'); ax[0].axis('off')
ax[1].imshow(binaria); ax[1].set_title('Máscara binaria'); ax[1].axis('off')
ax[2].imshow(etiquetas, cmap='tab20'); ax[2].set_title('Etiquetas'); ax[2].axis('off')
plt.show()

## Cuantificación (regionprops)

Una vez que completamos el etiquetado de los objetos en la imagen (labeled mask), podemos proceder a cuantificar atributos de estos objetos. Para ello utilizaremos la función regionprops de scikit-image.

In [ ]:
props = measure.regionprops_table(
    etiquetas,
    intensity_image=corregida,
    properties=['label', 'area', 'eccentricity', 'solidity', 'mean_intensity']
)
df = pd.DataFrame(props)
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df['area'], bins=25, color='tab:blue', alpha=0.8)
ax[0].set_title('Distribución de áreas')
ax[0].set_xlabel('Área (px)')
ax[0].set_ylabel('Frecuencia')

ax[1].scatter(df['area'], df['mean_intensity'], s=25, alpha=0.7, color='tab:purple')
ax[1].set_title('Área vs intensidad media')
ax[1].set_xlabel('Área (px)')
ax[1].set_ylabel('Intensidad media')

plt.tight_layout()
plt.show()

print(f'Objetos medidos: {len(df)}')

Con esta tabla podemos hacer análisis estadístico de los datos. Pero antes de eso, debemos ser críticos con nuestro flujo de análisis de datos y revisar que no hayan valores anormales u objetos a filtrar.